In [2]:
import pandas as pd

In [ ]:
df = pd.read_csv("InflAdj_Data_2019_2025-20th-20th.csv", parse_dates=['report_date'])
df = df[df['symbol'] != 'RNWF']
df = df.dropna(subset=['industry', 'report_date', 'close'])
df['report_date'] = pd.to_datetime(df['report_date'])

sector_daily = (
    df.groupby(['industry', pd.Grouper(key='report_date', freq='B')])['close']
    .mean()
)

sector_daily = sector_daily.reset_index(drop=False).merge(
    sector_daily.reset_index(drop=False)[['report_date', 'industry']]
    .groupby('industry').min().reset_index(),
    on='industry', suffixes=('', '_min')
).sort_values(['report_date_min', 'industry']).set_index(['industry', 'report_date'])

#AI helped with this solution as the original order was lost when unstacking
original_order = sector_daily.index.get_level_values('industry').unique()
df_rotated = sector_daily['close'].unstack('report_date').reindex(original_order)
sector_daily = df_rotated
sector_daily
sector_daily.to_csv("datasets/sector_daily.csv")

sector_mom = sector_daily.pct_change(periods=30, axis=1) * 100
sector_mom = sector_mom.dropna(how='all', axis=1)
sector_mom.to_csv("datasets/sector_mom.csv")

In [ ]:
df = pd.read_csv("InflAdj_Data_2019_2025-20th-20th.csv")

for symbol in ['AAPL', 'ADM', 'AAP', 'AGCO']:
    df_s = df[df['symbol'] == symbol]
    df_s.to_csv(f"datasets/df_{symbol}.csv")

In [ ]:
df = pd.read_csv("InflAdj_Data_2019_2025-20th-20th.csv")

df['report_quarter'] = pd.to_datetime(df['report_date']).dt.to_period('Q')
df['sector'] =df['sector'].fillna("Other")
df = df[~(df['symbol'].isna())]

df_lday = df.copy()
df_lday['report_quarter_shifted-TEMP'] = df_lday['report_quarter'].shift(-1)

def last_Qday(row):
    return row['report_quarter'] != row['report_quarter_shifted-TEMP']



df_lday.sort_values(by= ['symbol', 'report_date'], axis= 0, inplace= True)
df_lday['last_Qday'] = df_lday.apply(last_Qday, axis= 1)
df_lday.drop('report_quarter_shifted-TEMP', axis= 1, inplace= True)

df_lday_only = df_lday[df_lday['last_Qday'] == True]

avg_close_2019 = df[df['report_date'] == '2019-01-02'].groupby(by= ['report_date', 'sector']).agg({'close' : 'mean'})
avg_close_2019.rename({'close' : 'average close'}, axis= 1, inplace= True)
baseline_sector_avg = {item[0][1] : item[1] for item in avg_close_2019['average close'].items()}

df_grouped_lday_abs = df_lday_only.groupby(by= ['report_quarter', 'sector']).agg({'close' : 'mean'})
df_grouped_lday_abs.rename({'close' : 'average close'}, axis= 1, inplace= True)
quarters = set([k[0] for k in df_grouped_lday_abs['average close'].keys()])
sectors_lday_abs = [k[1] for k in df_grouped_lday_abs['average close'].keys()]
values_lday_abs = df_grouped_lday_abs['average close'].values
unique_sectors = sorted([s for s in df['sector'].unique()])
unique_sectors = unique_sectors[:8] + ['Real Estate', 'Technology', 'Utilities', 'Other']

avg_close_2019.to_csv("datasets/barplot/avg_close_2019-20th-20th.csv")
df_grouped_lday_abs.to_csv("datasets/barplot/df_grouped_lday_abs-20th-20th.csv")